# Presentation visualisations

A separate notebook for model comparisons, colonisation curves and development timelines. No demonstration results are used. The project reference folder was empty when this notebook was created: connect the real exports before interpreting any figures.

**Design:** consistent colours across figures, elapsed days on the growth and timeline axes, colonisation in percent, and self-contained interactive HTML exports. The palette below is provisional until the presentation theme is supplied.

## Input files
Place CSV files in `presentation_data/` at the project root. Missing files are skipped. Required columns:

| File | Columns |
|---|---|
| `model_metrics.csv` | dataset, split, model, metric, value, run |
| `observations.csv` | dataset, experiment, observation_id, genus, species, oyster_group, temperature_c, days, colonisation_pct |
| `baseline_curves.csv` | dataset, genus, species, oyster_group, temperature_c, days, predicted_pct, model |
| `heldout_predictions.csv` | dataset, split, model, observation_id, days, observed_pct, predicted_pct |
| `stages.csv` | dataset, experiment, temperature_c, species, stage, start_day, end_day |
| `events.csv` | dataset, experiment, temperature_c, species, event, day |

Use numeric days and temperatures (°C), and percentages in 0–100. `oyster_group` must be `cool`, `tropical`, or `not_oyster`, assigned from experimental metadata. Do not infer oyster categories from names. Baseline curves contain actual model predictions on a day grid, not interpolated observations. Export a single comparable substrate/strain condition per group, or extend the grouping keys.

Metrics must share evaluation protocols and units within each panel. Supply only out-of-sample predictions for diagnostics; repeated runs should be aggregated first or given additional grouping columns. Stage dates and events should be labelled as observed or estimated in the final slide captions.


In [ ]:
from pathlib import Path
import importlib.util
import warnings
missing = [m for m in ["pandas", "numpy", "plotly"] if importlib.util.find_spec(m) is None]
if missing:
    raise ImportError("Install notebook dependencies: " + ", ".join(missing))
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "presentation_data"
OUT = ROOT / "presentation_figures"
OUT.mkdir(exist_ok=True)
# Replace with the slide deck's actual theme colours.
PALETTE = ["#245A81", "#16877D", "#D18B29", "#8A659F", "#B95561", "#667B8C"]
TEXT, GRID, BACKGROUND = "#243746", "#E5EAF0", "#FFFFFF"
MODEL_COLOURS, STAGE_COLOURS = {}, {}  # Optional explicit overrides by name.
SHOW_FIGURES = True
EXPORT_PNG = False  # Requires Kaleido and its browser dependency.

def colours(names, overrides=None):
    names = sorted(set(map(str, names)))
    if len(names) > len(PALETTE):
        warnings.warn("Categories exceed palette size; specify distinct colours before presenting.")
    return {n: (overrides or {}).get(n, PALETTE[i % len(PALETTE)]) for i, n in enumerate(names)}

def finish(fig, title, filename, height=580):
    import re
    fig.update_layout(title=title, template="plotly_white", height=height,
        paper_bgcolor=BACKGROUND, plot_bgcolor=BACKGROUND,
        font=dict(family="Arial", size=16, color=TEXT),
        margin=dict(l=85, r=40, t=100, b=100), legend_title_text="")
    fig.update_xaxes(gridcolor=GRID)
    fig.update_yaxes(gridcolor=GRID)
    stem = re.sub(r"[^a-zA-Z0-9_-]+", "_", filename)
    fig.write_html(OUT / (stem + ".html"), include_plotlyjs=True)
    if EXPORT_PNG:
        fig.write_image(OUT / (stem + ".png"), width=1600, height=900, scale=2)
    if SHOW_FIGURES:
        fig.show()
    return fig


In [ ]:
from mycodemo.presentation_data import build_presentation_data

counts, skipped_curves = build_presentation_data(ROOT)
print(counts)
for item in skipped_curves:
    print("Curve skipped:", item)

In [ ]:
SCHEMAS = {
    "model_metrics": "dataset split model metric value run",
    "observations": "dataset experiment observation_id genus species oyster_group temperature_c days colonisation_pct",
    "baseline_curves": "dataset genus species oyster_group temperature_c days predicted_pct model",
    "heldout_predictions": "dataset split model observation_id days observed_pct predicted_pct",
    "stages": "dataset experiment temperature_c species stage start_day end_day",
    "events": "dataset experiment temperature_c species event day",
}
NUMERIC = {"value", "temperature_c", "days", "colonisation_pct", "observed_pct", "predicted_pct", "start_day", "end_day", "day"}
UNIQUE_KEYS = {
    "model_metrics": ["dataset", "split", "model", "metric", "run"],
    "observations": ["dataset", "observation_id"],
    "baseline_curves": ["dataset", "genus", "species", "oyster_group", "temperature_c", "days", "model"],
    "heldout_predictions": ["dataset", "split", "model", "observation_id"],
}

def load(name):
    columns = SCHEMAS[name].split()
    path = DATA / f"{name}.csv"
    if not path.exists():
        print(f"Awaiting {path}")
        return pd.DataFrame(columns=columns)
    df = pd.read_csv(path)
    absent = set(columns) - set(df.columns)
    if absent:
        raise ValueError(f"{name}: missing columns {sorted(absent)}")
    if df[columns].isna().any().any():
        raise ValueError(f"{name}: required values are missing")
    for col in set(columns) & NUMERIC:
        df[col] = pd.to_numeric(df[col], errors="raise")
        if not np.isfinite(df[col]).all():
            raise ValueError(f"{name}: {col} is not finite")
        if col.endswith("_pct") and not df[col].between(0, 100).all():
            raise ValueError(f"{name}: {col} is outside 0–100; investigate rather than clipping")
        if col in {"days", "start_day", "end_day", "day"} and (df[col] < 0).any():
            raise ValueError(f"{name}: negative elapsed days")
    if "oyster_group" in df and not df.oyster_group.isin(["cool", "tropical", "not_oyster"]).all():
        raise ValueError(f"{name}: invalid oyster_group")
    if name == "stages" and (df.end_day < df.start_day).any():
        raise ValueError("Stage ends before it starts")
    if name in UNIQUE_KEYS and df.duplicated(UNIQUE_KEYS[name]).any():
        raise ValueError(f"{name}: duplicate rows for {UNIQUE_KEYS[name]}")
    return df

tables = {name: load(name) for name in SCHEMAS}
model_names = set()
for df in tables.values():
    if "model" in df:
        model_names.update(df.model.astype(str))
model_palette = colours(model_names, MODEL_COLOURS)
print({name: len(df) for name, df in tables.items()})


## 1. Tested model performance
Separate panels prevent mixing datasets, splits or metrics. Individual runs/folds remain visible. Explain whether lower or higher is better in the caption; add uncertainty intervals only when they can be computed from a valid experimental unit. Coincident values can overlap.


In [ ]:
for (dataset, split, metric), g in tables["model_metrics"].groupby(["dataset", "split", "metric"], sort=True):
    fig = px.strip(g, x="value", y="model", color="model", hover_data=["run"],
                   color_discrete_map=model_palette)
    fig.update_traces(jitter=0)
    fig.update_xaxes(title=str(metric))
    fig.update_yaxes(title="Model")
    finish(fig, f"{dataset} · {split} · {metric}", f"metrics_{dataset}_{split}_{metric}")


## 2. Colonisation observations against baseline predictions
One plot per dataset/genus, with cool and tropical oysters split further. The slider selects temperatures without interpolating across them. Observations are points; predictions are lines. Axis ranges stay fixed when the slider changes. Species use different point symbols and labelled lines.

Baseline curves: S-shape from the logistic growth model (literature); midpoint is a demo assumption.


In [ ]:
obs, curves = tables["observations"], tables["baseline_curves"]
keys = ["dataset", "genus", "oyster_group"]
contexts = pd.concat([obs[keys], curves[keys]]).drop_duplicates()
for dataset, genus, oyster_group in contexts.itertuples(index=False, name=None):
    def select(df):
        return df[(df.dataset == dataset) & (df.genus == genus) & (df.oyster_group == oyster_group)]
    o, c = select(obs), select(curves)
    temperatures = sorted(set(o.temperature_c) | set(c.temperature_c))
    species = sorted(set(o.species) | set(c.species))
    symbols = {s: ["circle", "square", "diamond", "triangle-up", "cross", "x"][i % 6] for i, s in enumerate(species)}
    dashes = {s: ["solid", "dash", "dot", "dashdot", "longdash", "longdashdot"][i % 6] for i, s in enumerate(species)}
    fig, trace_temps = go.Figure(), []
    for temp in temperatures:
        for (sp, experiment), g in o[o.temperature_c == temp].groupby(["species", "experiment"]):
            fig.add_trace(go.Scatter(x=g.days, y=g.colonisation_pct, mode="markers",
                name=f"{sp} · {experiment} (observed)",
                marker=dict(symbol=symbols[sp], size=9, color=TEXT, opacity=.75),
                customdata=g[["observation_id"]].astype(str).to_numpy(),
                hovertemplate="Day %{x}<br>%{y:.1f}%<br>ID %{customdata[0]}<extra>%{fullData.name}</extra>"))
            trace_temps.append(temp)
        for (sp, model), g in c[c.temperature_c == temp].groupby(["species", "model"]):
            g = g.sort_values("days")
            fig.add_trace(go.Scatter(x=g.days, y=g.predicted_pct, mode="lines", name=f"{sp} · {model}",
                line=dict(color=model_palette[str(model)], width=3, dash=dashes[sp])))
            trace_temps.append(temp)
    for trace, temp in zip(fig.data, trace_temps):
        trace.visible = temp == temperatures[0]
    suffix = "" if oyster_group == "not_oyster" else f" · {oyster_group} oysters"
    title = f"{dataset} · {genus}{suffix}"
    fig.update_layout(sliders=[dict(active=0, currentvalue=dict(prefix="Temperature °C: "),
        steps=[dict(label=f"{temp:g}", method="update", args=[
            {"visible": [t == temp for t in trace_temps]},
            {"title": {"text": title + f" · {temp:g} °C"}}]) for temp in temperatures])])
    all_days = pd.concat([o.days, c.days])
    fig.update_xaxes(title="Days", range=[0, max(float(all_days.max()), 1)*1.03])
    fig.update_yaxes(title="Colonisation (%)", range=[0, 100])
    finish(fig, title + f" · {temperatures[0]:g} °C", f"colonisation_{dataset}_{genus}_{oyster_group}")


## 3. Development stages and events
Each species has a row. Stage intervals use consistent colours and events use dark diamond dots with hover labels. Conditions are separated by dataset, experiment and temperature. Overlapping stages remain overlapping; use separate lanes if real data makes this hard to read. Zero-duration stages should be represented as events.


In [ ]:
stages, events = tables["stages"], tables["events"]
stage_palette = colours(stages.stage, STAGE_COLOURS)
keys = ["dataset", "experiment", "temperature_c"]
contexts = pd.concat([stages[keys], events[keys]]).drop_duplicates()
for dataset, experiment, temp in contexts.itertuples(index=False, name=None):
    def select(df):
        return df[(df.dataset == dataset) & (df.experiment == experiment) & (df.temperature_c == temp)]
    s, e = select(stages), select(events)
    fig = go.Figure()
    for stage, g in s.groupby("stage", sort=True):
        fig.add_trace(go.Bar(y=g.species, x=g.end_day-g.start_day, base=g.start_day,
            orientation="h", name=str(stage), marker_color=stage_palette[str(stage)],
            customdata=g[["start_day", "end_day"]].to_numpy(),
            hovertemplate="%{y}<br>%{customdata[0]}–%{customdata[1]} days<extra>%{fullData.name}</extra>"))
    for event, g in e.groupby("event", sort=True):
        fig.add_trace(go.Scatter(x=g.day, y=g.species, mode="markers", name=str(event),
            marker=dict(symbol="diamond", size=13, color=TEXT, line=dict(color="white", width=1)),
            hovertemplate="%{y}<br>Day %{x}<extra>%{fullData.name}</extra>"))
    species = sorted(set(s.species) | set(e.species))
    fig.update_layout(barmode="overlay")
    fig.update_xaxes(title="Days", rangemode="tozero")
    fig.update_yaxes(title="Species", categoryorder="array", categoryarray=species, autorange="reversed")
    finish(fig, f"Development · literature typical values · {dataset} · {experiment} · {temp:g} °C",
           f"timeline_{dataset}_{experiment}_{temp}", height=max(450, 150+65*len(species)))


## 4. Suggested addition: prediction accuracy and errors over time
Observed versus predicted values show calibration against the dashed identity line. Residuals show overprediction (positive) or underprediction (negative), and systematic errors across days. Use held-out experiments where possible to avoid leakage from repeated observations.


In [ ]:
pred = tables["heldout_predictions"].copy()
if not pred.empty:
    pred["residual_pct"] = pred.predicted_pct - pred.observed_pct
    for (dataset, split), g in pred.groupby(["dataset", "split"]):
        fig = px.scatter(g, x="observed_pct", y="predicted_pct", color="model",
            hover_data=["observation_id", "days"], color_discrete_map=model_palette)
        fig.add_shape(type="line", x0=0, x1=100, y0=0, y1=100, line=dict(color=TEXT, dash="dash"))
        fig.update_xaxes(title="Observed colonisation (%)", range=[0, 100])
        fig.update_yaxes(title="Predicted colonisation (%)", range=[0, 100], scaleanchor="x", scaleratio=1)
        finish(fig, f"Held-out predictions · {dataset} · {split}", f"predicted_observed_{dataset}_{split}")
        fig = px.scatter(g, x="days", y="residual_pct", color="model",
            hover_data=["observation_id"], color_discrete_map=model_palette)
        fig.add_hline(y=0, line_dash="dash", line_color=TEXT)
        fig.update_xaxes(title="Days")
        fig.update_yaxes(title="Prediction − observation (percentage points)")
        finish(fig, f"Residuals · {dataset} · {split}", f"residuals_{dataset}_{split}")


## Presentation checklist and further improvements
- Replace the provisional theme and connect the real exports. No scientific results have been verified yet.
- Add sample sizes, data sources, metric units/direction and evaluation split definitions to slide captions.
- Add a temperature-by-species coverage heatmap if sampling is uneven; it helps explain gaps.
- Add replicate-level confidence bands when the experimental design supports them. Do not count repeated daily measurements as independent replicates.
- Identify observed, estimated and censored stage dates. Unknown dates must not become zero.
- For static slides, choose one temperature and use Plotly's camera button. Interactive HTML retains the slider. Optional PNG export requires Kaleido and its browser dependency.

Figures export to `presentation_figures/`. This notebook is an input-ready starting point; wiring it to existing training notebooks and matching the actual deck requires the missing project files.
